# LeetCode 361 — Bomb Enemy bằng Python

Link đề: https://leetcode.com/problems/bomb-enemy/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Matrix (lưới 2D), Prefix / Running Count (đếm cộng dồn)

> **Bài Premium (🔒).** Đề trong notebook này được viết theo hiểu biết của mình về bài và theo bản mô tả đã có; nếu bạn có tài khoản Premium, hãy **đối chiếu lại đề gốc** (nhất là ví dụ, và trường hợp lưới không có ô trống nào) trước khi nộp.

Skeleton LeetCode:

```python
class Solution:
    def maxKilledEnemies(self, grid: List[List[str]]) -> int:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu năm cách làm với một oracle độc lập trên hàng nghìn lưới ngẫu nhiên). Ví dụ nào ghi "tự dựng" là do mình thêm để minh họa — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lưới `m × n` gồm tường `'W'`, kẻ địch `'E'`, ô trống `'0'` (chữ số không **dạng ký tự**); đặt **một quả bom** vào **một ô trống**; bom nổ theo **hàng và cột** của ô đó (bốn hướng) cho đến khi **gặp tường**; hỏi **tối đa** diệt được bao nhiêu kẻ địch.
2. Nắm các công cụ Python: lưới là `list[list[str]]` (so sánh **ký tự** `'0'`, không phải số `0`), hai vòng `for i in range(m): for j in range(n)`, `max(..., default)`, **chuyển vị** `zip(*grid)`, `str.split('W')` / `str.count('E')`, và **biến đếm cộng dồn** được "reset" khi gặp tường.
3. Đi từ cách **đặt bom ở mỗi ô rồi quét bốn hướng** (`O(mn(m+n))`) → **bốn bảng quy hoạch động** (đếm kẻ địch về bốn phía) → **đếm cộng dồn tái sử dụng** (`O(mn)` thời gian, `O(n)` bộ nhớ thêm) → **chia đoạn theo tường bằng `split`** → **cách ngược**: từ **mỗi kẻ địch** "cộng điểm" cho các ô trống bị nó bắn tới.
4. Hiểu mấu chốt: số kẻ địch bị diệt bởi bom ở `(i, j)` = (số `E` trong **đoạn hàng** chứa `(i, j)` giữa hai bức tường) + (số `E` trong **đoạn cột** chứa `(i, j)`). Đoạn hàng chỉ cần **đếm một lần** khi bước vào đầu đoạn (sau tường hoặc đầu hàng) và **dùng lại** cho mọi ô trong đoạn.
5. Tránh các lỗi kinh điển: **quên tường chặn**, đặt bom lên ô **`'E'`** hay **`'W'`**, **không reset** bộ đếm tại tường, so sánh ô với **`0` (số)** thay vì **`'0'` (chuỗi)**, chỉ đếm hai trong bốn hướng, và khởi tạo kết quả sai khi **không có ô trống** nào.

## 2. Đề bài & ví dụ

Cho lưới `grid` kích thước `m × n`, mỗi ô là một trong ba ký tự:

- `'W'`: **tường**,
- `'E'`: **kẻ địch**,
- `'0'`: **ô trống** (chữ số không dạng ký tự).

Trả về **số kẻ địch tối đa** bạn có thể diệt bằng **một quả bom**. Bom chỉ được đặt vào **ô trống**. Bom diệt **mọi kẻ địch cùng hàng và cùng cột** với điểm đặt, tính từ điểm đặt đi ra bốn phía **cho tới khi gặp tường** (bom quá mạnh nên không bị kẻ địch cản, nhưng tường thì chặn lại).

**Ví dụ 1**

- Input: `grid = [["0","E","0","0"],["E","0","W","E"],["0","E","0","0"]]`
- Output: `3`
- Giải thích: đặt bom ở ô `(1, 1)` (hàng `1`, cột `1`): hàng có `E` ở `(1, 0)` (bên phải gặp tường `W` ở `(1, 2)` nên dừng); cột có `E` ở `(0, 1)` và `(2, 1)` ⇒ `1 + 2 = 3`.

**Ví dụ 2**

- Input: `grid = [["W","W","W"],["0","0","0"],["E","E","E"]]`
- Output: `1`
- Giải thích: đặt bom ở một ô hàng giữa: hàng giữa không có `E`; cột của ô đó có tường `W` ở trên và `E` ở dưới ⇒ mỗi cột diệt `1` kẻ địch.

**Ràng buộc:**

- `m == grid.length`, `n == grid[i].length`
- `1 <= m, n <= 500`
- `grid[i][j]` là `'W'`, `'E'` hoặc `'0'`.

In [1]:
import random
import time
from collections import Counter
from typing import List

EX1 = [["0", "E", "0", "0"], ["E", "0", "W", "E"], ["0", "E", "0", "0"]]
EX2 = [["W", "W", "W"], ["0", "0", "0"], ["E", "E", "E"]]
examples = [(EX1, 3), (EX2, 1)]

for g, expected in examples:
    print("\n".join(" ".join(row) for row in g), "-> kỳ vọng", expected, "\n")

0 E 0 0
E 0 W E
0 E 0 0 -> kỳ vọng 3 

W W W
0 0 0
E E E -> kỳ vọng 1 



## 3. Kiến thức nền

### 3.1. Lưới là list của list; ký tự `'0'` khác số `0`

`grid[i][j]` là ký tự ở hàng `i`, cột `j`. Mỗi ô là **chuỗi một ký tự**: `'W'`, `'E'` hoặc `'0'`. **Chú ý:** `'0'` (ký tự chữ số không) **khác** `0` (số nguyên): `'0' == 0` là `False`. Vì vậy so sánh ô trống phải viết `grid[i][j] == '0'`.

In [2]:
grid = EX1
m, n = len(grid), len(grid[0])
print(m, n, grid[1][3], grid[1][1])
print("'0' == 0 ?", '0' == 0, "| '0' == '0' ?", '0' == '0')
print("số ô trống:", sum(row.count('0') for row in grid), "| số E:", sum(row.count('E') for row in grid))

3 4 E 0
'0' == 0 ? False | '0' == '0' ? True
số ô trống: 7 | số E: 4


### 3.2. Duyệt lưới và bốn hướng

Hai vòng lặp `for i in range(m): for j in range(n)` duyệt mọi ô. Bốn hướng đi: `(0, 1)` phải, `(0, −1)` trái, `(1, 0)` xuống, `(−1, 0)` lên — gọi chung `(di, dj)`. Từ `(i, j)` đi theo hướng `(di, dj)`: `(i + di, j + dj)`, rồi `(i + 2di, j + 2dj)`, ... cho đến khi **ra ngoài lưới** hoặc **gặp tường**.

In [3]:
DIRS = [(0, 1), (0, -1), (1, 0), (-1, 0)]

def ray(grid, i, j, di, dj):
    """Các ô mà bom ở (i, j) bắn tới theo hướng (di, dj), dừng trước tường."""
    cells = []
    x, y = i + di, j + dj
    while 0 <= x < len(grid) and 0 <= y < len(grid[0]) and grid[x][y] != 'W':
        cells.append((x, y))
        x, y = x + di, y + dj
    return cells

for d in DIRS:
    print(d, ray(EX1, 1, 1, *d))

(0, 1) []
(0, -1) [(1, 0)]
(1, 0) [(2, 1)]
(-1, 0) [(0, 1)]


### 3.3. Bộ đếm cộng dồn (running count) có reset

Khi quét một hàng từ trái sang phải, ta giữ biến `cnt` = "số `E` đã gặp **kể từ bức tường gần nhất bên trái**". Gặp `'E'` thì `cnt += 1`; gặp `'W'` thì **reset** `cnt = 0`. Ý tưởng "đếm cộng dồn, reset ở chướng ngại" xuất hiện rất nhiều ở bài lưới/mảng.

In [4]:
row = list("0E0W0EE0")
cnt = 0
for j, ch in enumerate(row):
    if ch == 'W':
        cnt = 0
    elif ch == 'E':
        cnt += 1
    print(j, ch, "cnt =", cnt)

0 0 cnt = 0
1 E cnt = 1
2 0 cnt = 1
3 W cnt = 0
4 0 cnt = 0
5 E cnt = 1
6 E cnt = 2
7 0 cnt = 2


### 3.4. Chuyển vị `zip(*grid)` để xử lý cột như hàng

`zip(*grid)` "xoay" lưới: hàng thứ `j` của kết quả là **cột `j`** của lưới. Nhờ đó mọi thuật toán cho **hàng** dùng lại được cho **cột**.

In [5]:
cols = list(zip(*EX1))
print(cols)                         # mỗi phần tử là một tuple: cột j
print("".join(cols[1]), "".join(cols[2]))     # 'E0E' và '0W0'

[('0', 'E', '0'), ('E', '0', 'E'), ('0', 'W', '0'), ('0', 'E', '0')]
E0E 0W0


### 3.5. `str.split('W')` và `str.count('E')`

Nếu ghép một hàng thành chuỗi `s = "".join(row)` thì `s.split('W')` **cắt** chuỗi tại mỗi tường thành các **đoạn** (các đoạn có thể rỗng). `đoạn.count('E')` cho số kẻ địch trong đoạn. Các đoạn chính là "tầm nổ" giữa hai tường.

In [6]:
s = "".join(["0", "E", "0", "W", "0", "E", "E"])
print(s, "->", s.split('W'), "->", [seg.count('E') for seg in s.split('W')])
print("WW0".split('W'), "0W".split('W'))        # đoạn rỗng xuất hiện khi tường kề nhau/ở biên

0E0W0EE -> ['0E0', '0EE'] -> [1, 2]
['', '', '0'] ['0', '']


### 3.6. `max` với danh sách có thể rỗng

Nếu không có ô trống nào thì không có ứng viên nào cho `max`; `max([])` báo `ValueError`. Dùng `max(..., default=0)` hoặc khởi tạo `best = 0` rồi cập nhật bằng `best = max(best, giá_trị)`.

In [7]:
try:
    max([])
except ValueError as e:
    print("max([]) ->", e)
print(max([], default=0), max([3, 5], default=0))
best = 0
for v in []:
    best = max(best, v)
print("best =", best)

max([]) -> max() iterable argument is empty
0 5
best = 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với **Ví dụ 1**, thử đặt bom ở từng ô trống và đếm:

```
      c0 c1 c2 c3
r0:   0  E  0  0
r1:   E  0  W  E
r2:   0  E  0  0
```

| ô trống | hàng (hai phía, tới tường) | cột (hai phía, tới tường) | tổng |
|---|---|---|---|
| `(0,0)` | phải: `E` ở `(0,1)` ⇒ `1` | xuống: `E` ở `(1,0)` ⇒ `1` | `2` |
| `(0,2)` | trái: `E` ở `(0,1)` ⇒ `1`; phải: không | xuống: `(1,2)` là `W` ⇒ chặn ⇒ `0` | `1` |
| `(0,3)` | trái: `E` ở `(0,1)` ⇒ `1` | xuống: `E` ở `(1,3)` ⇒ `1` | `2` |
| `(1,1)` | trái: `E` ở `(1,0)` ⇒ `1`; phải: `W` chặn | lên: `E` ở `(0,1)` ⇒ `1`; xuống: `E` ở `(2,1)` ⇒ `1` | **`3`** |
| `(2,0)` | phải: `E` ở `(2,1)` ⇒ `1` | lên: `E` ở `(1,0)` ⇒ `1` | `2` |
| `(2,2)` | trái: `E` ở `(2,1)` ⇒ `1` | lên: `(1,2)` là `W` ⇒ `0` | `1` |
| `(2,3)` | trái: `E` ở `(2,1)` ⇒ `1` | lên: `E` ở `(1,3)` ⇒ `1` | `2` |

Lớn nhất là `3` ở ô `(1, 1)` ✔.

**Quan sát:** để tính một ô ta **quét** bốn hướng — các ô cùng hàng (cùng đoạn) quét **lặp lại** gần như cùng một đoạn. Tối ưu: tính số `E` của **mỗi đoạn một lần**.

In [8]:
def kills_at(grid, i, j):
    """Số kẻ địch bị diệt khi đặt bom ở (i, j): quét bốn hướng (đang dùng 'ray' đã định nghĩa)."""
    return sum(1 for d in DIRS for (x, y) in ray(grid, i, j, *d) if grid[x][y] == 'E')


for i in range(3):
    for j in range(4):
        if EX1[i][j] == '0':
            print((i, j), "->", kills_at(EX1, i, j))

(0, 0) -> 2
(0, 2) -> 1
(0, 3) -> 2
(1, 1) -> 3
(2, 0) -> 2
(2, 2) -> 1
(2, 3) -> 2


## 5. Cách 1 — Brute Force / baseline: đặt bom ở mỗi ô trống rồi quét bốn hướng

**Ý tưởng.** Với **mỗi ô trống**, quét lần lượt bốn hướng, đếm `E` cho đến khi gặp tường hoặc ra ngoài lưới. Lấy tối đa.

In [9]:
def max_killed_bruteforce(grid):
    m, n = len(grid), len(grid[0])
    best = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] != '0':
                continue
            total = 0
            for di, dj in DIRS:
                x, y = i + di, j + dj
                while 0 <= x < m and 0 <= y < n and grid[x][y] != 'W':
                    if grid[x][y] == 'E':
                        total += 1
                    x += di
                    y += dj
            best = max(best, total)
    return best


for g, expected in examples:
    print(max_killed_bruteforce(g), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )


**Độ phức tạp (Cách 1).** Mỗi ô trống quét tối đa `m + n` ô ⇒ **Time `O(m·n·(m + n))`** (với `m = n = 500`: khoảng `500 · 500 · 1000 = 2.5 · 10^8` bước — quá chậm trong Python); **Space `O(1)`** thêm.

**Nút thắt:** các ô **cùng đoạn hàng** đều quét lại đúng đoạn đó. Ví dụ hàng `"0 E 0 0 E 0"` (không tường): cả `4` ô trống đều đếm lại `2` kẻ địch của cùng một hàng. Nên đếm **một lần cho cả đoạn**.

In [10]:
# Minh họa nút thắt: lưới 150x150 toàn ô trống/kẻ địch, không tường -> mỗi ô quét 300 ô
rng = random.Random(1)
g = [[rng.choice("0E") for _ in range(150)] for _ in range(150)]
t0 = time.perf_counter()
max_killed_bruteforce(g)
print(f"Cách 1 trên 150x150: {time.perf_counter() - t0:.2f}s  (500x500 sẽ lâu hơn ~ (500/150)^3 ≈ 37 lần)")

Cách 1 trên 150x150: 0.44s  (500x500 sẽ lâu hơn ~ (500/150)^3 ≈ 37 lần)


## 6. Ý tưởng thuật toán / bước đột phá

Gọi `rowSeg(i, j)` là **đoạn hàng** chứa `(i, j)` (các ô liên tiếp cùng hàng `i` giữa hai bức tường hoặc biên, **không gồm tường**), và `colSeg(i, j)` tương tự theo cột. Thì:

> số diệt được tại `(i, j)` = `E(rowSeg(i, j)) + E(colSeg(i, j))`,

với `E(đoạn)` là số `'E'` trong đoạn. (Ô `(i, j)` là ô trống nên không đóng góp thêm.)

Hai cách tính `E(đoạn)` mà không quét lại:

- **Quy hoạch động bốn hướng:** `left[i][j]` = số `E` bên trái liên tiếp tới tường; tương tự `right`, `up`, `down`. Mỗi bảng tính trong `O(mn)` bằng công thức truy hồi.
- **Đếm tái sử dụng:** khi bước vào **ô đầu đoạn hàng** (cột `0` hoặc ô bên trái là `W`), quét **một lần** tới hết đoạn để tính `row_hits`; mọi ô còn lại trong đoạn dùng lại `row_hits`. Làm tương tự cho cột với mảng `col_hits[j]`. Mỗi ô bị quét tối đa **một lần theo hàng** và **một lần theo cột** ⇒ `O(mn)`.

## 7. Các cách tiếp theo

### Cách 2 — Bốn bảng quy hoạch động (trái, phải, trên, dưới)

**Giải thích từng đoạn.**

1. `left[i][j]` = số `E` trong các ô `(i, j − 1), (i, j − 2), ...` cho đến tường (không tính ô `(i, j)`). Truy hồi: `j == 0` ⇒ `0`; ô trái là `'W'` ⇒ `0`; ô trái là `'E'` ⇒ `left[i][j − 1] + 1`; ô trái là `'0'` ⇒ `left[i][j − 1]`.
2. `right`, `up`, `down` tính tương tự theo hướng riêng (duyệt ngược để ô "bên kia" đã có giá trị).
3. Với mỗi ô trống: `left + right + up + down`; lấy tối đa.

Bảng `left` thứ `i, j` dựa vào ô trái đã tính (duyệt `j` tăng); `right` duyệt `j` giảm; `up` duyệt `i` tăng; `down` duyệt `i` giảm.

In [11]:
def max_killed_dp(grid):
    m, n = len(grid), len(grid[0])
    left = [[0] * n for _ in range(m)]
    right = [[0] * n for _ in range(m)]
    up = [[0] * n for _ in range(m)]
    down = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(1, n):                       # trái: duyệt j tăng
            p = grid[i][j - 1]
            left[i][j] = 0 if p == 'W' else left[i][j - 1] + (p == 'E')
        for j in range(n - 2, -1, -1):              # phải: duyệt j giảm
            p = grid[i][j + 1]
            right[i][j] = 0 if p == 'W' else right[i][j + 1] + (p == 'E')
    for j in range(n):
        for i in range(1, m):                       # trên: duyệt i tăng
            p = grid[i - 1][j]
            up[i][j] = 0 if p == 'W' else up[i - 1][j] + (p == 'E')
        for i in range(m - 2, -1, -1):              # dưới: duyệt i giảm
            p = grid[i + 1][j]
            down[i][j] = 0 if p == 'W' else down[i + 1][j] + (p == 'E')
    best = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == '0':
                best = max(best, left[i][j] + right[i][j] + up[i][j] + down[i][j])
    return best


for g, expected in examples:
    print(max_killed_dp(g), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionDP:
    def maxKilledEnemies(self, grid: List[List[str]]) -> int:
        m, n = len(grid), len(grid[0])
        left = [[0] * n for _ in range(m)]
        right = [[0] * n for _ in range(m)]
        up = [[0] * n for _ in range(m)]
        down = [[0] * n for _ in range(m)]
        for i in range(m):
            for j in range(1, n):
                p = grid[i][j - 1]
                left[i][j] = 0 if p == 'W' else left[i][j - 1] + (p == 'E')
            for j in range(n - 2, -1, -1):
                p = grid[i][j + 1]
                right[i][j] = 0 if p == 'W' else right[i][j + 1] + (p == 'E')
        for j in range(n):
            for i in range(1, m):
                p = grid[i - 1][j]
                up[i][j] = 0 if p == 'W' else up[i - 1][j] + (p == 'E')
            for i in range(m - 2, -1, -1):
                p = grid[i + 1][j]
                down[i][j] = 0 if p == 'W' else down[i + 1][j] + (p == 'E')
        best = 0
        for i in range(m):
            for j in range(n):
                if grid[i][j] == '0':
                    best = max(best, left[i][j] + right[i][j] + up[i][j] + down[i][j])
        return best


print(SolutionDP().maxKilledEnemies(EX1))     # 3

3


**Độ phức tạp Cách 2:** bốn bảng `m × n`, mỗi ô tính `O(1)` ⇒ **Time `O(m·n)`**; **Space `O(m·n)`** (bốn bảng; có thể gộp thành một bảng cộng dồn — xem Cách 5 ý tương tự).

### Cách 3 — Đếm tái sử dụng: `row_hits` và `col_hits[j]` (cách chuẩn, `O(n)` bộ nhớ thêm)

**Ý tưởng.** Duyệt lưới theo hàng, trái sang phải. Giữ:

- `row_hits`: số `E` trong **đoạn hàng hiện tại**; **tính lại** khi bước vào đầu đoạn: `j == 0` hoặc `grid[i][j − 1] == 'W'`.
- `col_hits[j]`: số `E` trong **đoạn cột hiện tại của cột `j`**; **tính lại** khi `i == 0` hoặc `grid[i − 1][j] == 'W'`.

"Tính lại" nghĩa là quét một đoạn tới tường (hoặc biên) bằng vòng `while`. Sau đó nếu `grid[i][j] == '0'` thì ứng viên là `row_hits + col_hits[j]`.

**Giải thích từng đoạn code.**

1. `row_hits = 0`, `col_hits = [0] * n`, `best = 0`.
2. Với mỗi `(i, j)`: nếu là đầu đoạn hàng thì quét từ `(i, j)` sang phải tới khi gặp `'W'` hoặc hết hàng, đếm `E`.
3. Nếu là đầu đoạn cột thì quét từ `(i, j)` xuống tới khi gặp `'W'` hoặc hết cột, đếm `E`.
4. Nếu ô là `'0'`: `best = max(best, row_hits + col_hits[j])`.

Ô `'W'` không phải ô trống nên không tạo ứng viên; ô ngay bên phải (hoặc ngay bên dưới) nó sẽ là **đầu đoạn mới** và kích hoạt quét lại.

In [13]:
def max_killed_cached(grid):
    m, n = len(grid), len(grid[0])
    row_hits = 0
    col_hits = [0] * n
    best = 0
    for i in range(m):
        for j in range(n):
            if j == 0 or grid[i][j - 1] == 'W':          # đầu đoạn hàng: đếm E tới tường/biên phải
                row_hits = 0
                k = j
                while k < n and grid[i][k] != 'W':
                    row_hits += grid[i][k] == 'E'
                    k += 1
            if i == 0 or grid[i - 1][j] == 'W':          # đầu đoạn cột: đếm E tới tường/biên dưới
                col_hits[j] = 0
                k = i
                while k < m and grid[k][j] != 'W':
                    col_hits[j] += grid[k][j] == 'E'
                    k += 1
            if grid[i][j] == '0':
                best = max(best, row_hits + col_hits[j])
    return best


for g, expected in examples:
    print(max_killed_cached(g), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )


**Độ phức tạp Cách 3:** mỗi ô bị quét **tối đa một lần** bởi vòng `while` của đoạn hàng (chỉ quét khi ở **đầu** đoạn) và **tối đa một lần** bởi vòng `while` của đoạn cột ⇒ tổng `O(mn)` ⇒ **Time `O(m·n)`**; **Space `O(n)`** (mảng `col_hits`). Đây là cách chuẩn.

Một chi tiết dễ vướng: nếu `grid[i][j]` là `'W'` thì ô bên phải (nếu có) sẽ thấy "ô trái là `W`" và bắt đầu đoạn mới. Nếu ô đầu đoạn lại chính là `'W'` (hai tường liên tiếp) thì vòng `while` thoát ngay và `row_hits = 0` — đúng, vì ô tường không phải ô trống nên không dùng giá trị này.

### Cách 4 — Chia đoạn bằng `split('W')` cho hàng và cột (thao tác chuỗi)

**Ý tưởng.** Ghép mỗi hàng thành chuỗi, cắt bằng `split('W')`, đếm `E` mỗi đoạn, rồi "trải" kết quả về từng ô. Cột: dùng `zip(*grid)` để lấy cột như một hàng. Cuối cùng ghép `rows_kill[i][j] + cols_kill[j][i]`.

**Giải thích từng đoạn.**

1. Hàm `seg_counts(line)`: với chuỗi `line`, mỗi đoạn `seg` của `line.split('W')` có `e = seg.count('E')`; mọi ô trong đoạn nhận giá trị `e` (`[e] * len(seg)`), và mỗi chữ `'W'` giữa các đoạn nhận `0`.
2. `rows_kill[i] = seg_counts("".join(grid[i]))`; `cols_kill[j] = seg_counts("".join(cột j))`.
3. `best = max` trên ô trống của `rows_kill[i][j] + cols_kill[j][i]`.

In [14]:
def seg_counts(line):
    """Với chuỗi line, trả list: mỗi vị trí = số E của đoạn (giữa hai 'W') chứa vị trí đó; ô 'W' nhận 0."""
    res = []
    for seg in line.split('W'):
        e = seg.count('E')
        res.extend([e] * len(seg))
        res.append(0)                    # vị trí của chữ 'W' ngăn cách (sẽ bỏ cái thừa ở cuối)
    res.pop()
    return res


print(seg_counts("0E0W0EE0"), seg_counts("W0W"), seg_counts("EE"))


def max_killed_split(grid):
    m, n = len(grid), len(grid[0])
    rows_kill = [seg_counts("".join(row)) for row in grid]
    cols_kill = [seg_counts("".join(col)) for col in zip(*grid)]
    best = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == '0':
                best = max(best, rows_kill[i][j] + cols_kill[j][i])
    return best


for g, expected in examples:
    print(max_killed_split(g), "(kỳ vọng", expected, ")")

[1, 1, 1, 0, 2, 2, 2, 2] [0, 0, 0] [2, 2]
3 (kỳ vọng 3 )
1 (kỳ vọng 1 )


In [15]:
# Code nộp LeetCode cho Cách 4
class SolutionSplit:
    def maxKilledEnemies(self, grid: List[List[str]]) -> int:
        def seg_counts(line: str) -> List[int]:
            res = []
            for seg in line.split('W'):
                e = seg.count('E')
                res.extend([e] * len(seg))
                res.append(0)
            res.pop()
            return res

        m, n = len(grid), len(grid[0])
        rows_kill = [seg_counts("".join(row)) for row in grid]
        cols_kill = [seg_counts("".join(col)) for col in zip(*grid)]
        best = 0
        for i in range(m):
            for j in range(n):
                if grid[i][j] == '0':
                    best = max(best, rows_kill[i][j] + cols_kill[j][i])
        return best


print(SolutionSplit().maxKilledEnemies(EX2))     # 1

1


**Độ phức tạp Cách 4:** `join`, `split`, `count` đều tuyến tính theo độ dài chuỗi ⇒ **Time `O(m·n)`**; **Space `O(m·n)`** (hai bảng đếm, các chuỗi tạm). Chạy nhanh trong Python nhờ các hàm chuỗi viết bằng C.

### Cách 5 — Cách "ngược": từ mỗi kẻ địch cộng điểm cho các ô trống nó bắn tới

**Ý tưởng.** Thay vì "với mỗi ô trống hỏi có bao nhiêu kẻ địch bắn tới", ta làm ngược: **với mỗi kẻ địch**, đi bốn hướng tới khi gặp tường; **mỗi ô trống** gặp trên đường nhận `+1` điểm (`g[ô] += 1`). Cuối cùng `max` điểm trên các ô trống. Tương đương vì quan hệ "bom ở ô X diệt địch Y" là **đối xứng** (cùng hàng/cột và không có tường giữa hai ô).

**Giải thích từng đoạn.**

1. `g = [[0] * n for _ in range(m)]`.
2. Với mỗi ô `(i, j)` là `'E'`: với mỗi hướng, đi tới khi gặp `'W'`/ra ngoài; ô nào là `'0'` thì `g[x][y] += 1`. (Đi **qua** các `'E'` khác vì bom không bị địch cản.)
3. `best = max(g[i][j] cho các ô '0')`, mặc định `0`.

In [16]:
def max_killed_from_enemies(grid):
    m, n = len(grid), len(grid[0])
    g = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if grid[i][j] != 'E':
                continue
            for di, dj in DIRS:
                x, y = i + di, j + dj
                while 0 <= x < m and 0 <= y < n and grid[x][y] != 'W':
                    if grid[x][y] == '0':
                        g[x][y] += 1             # ô trống (x, y) bị kẻ địch (i, j) "nhìn thấy"
                    x += di
                    y += dj
    return max((g[i][j] for i in range(m) for j in range(n) if grid[i][j] == '0'), default=0)


for g_, expected in examples:
    print(max_killed_from_enemies(g_), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )


In [17]:
# Code nộp LeetCode cho Cách 5
class SolutionFromEnemies:
    def maxKilledEnemies(self, grid: List[List[str]]) -> int:
        m, n = len(grid), len(grid[0])
        g = [[0] * n for _ in range(m)]
        for i in range(m):
            for j in range(n):
                if grid[i][j] != 'E':
                    continue
                for di, dj in ((0, 1), (0, -1), (1, 0), (-1, 0)):
                    x, y = i + di, j + dj
                    while 0 <= x < m and 0 <= y < n and grid[x][y] != 'W':
                        if grid[x][y] == '0':
                            g[x][y] += 1
                        x += di
                        y += dj
        return max((g[i][j] for i in range(m) for j in range(n) if grid[i][j] == '0'), default=0)


print(SolutionFromEnemies().maxKilledEnemies(EX1))     # 3

3


**Độ phức tạp Cách 5:** mỗi kẻ địch quét tối đa `m + n` ô ⇒ **Time `O(E · (m + n))`** với `E` là số kẻ địch (tệ nhất `O(m·n·(m + n))` khi gần như toàn `E` mà vẫn có vài ô trống, nhưng thường tốt khi kẻ địch **ít**, kém hơn Cách 3 khi kẻ địch nhiều); **Space `O(m·n)`**. Hữu ích khi kẻ địch **thưa** và ô trống nhiều. Ý tưởng "đảo vai": **cộng điểm từ nguồn** thay vì **thu thập ở đích**.

## 8. TRACE từng bước

Trace **Cách 3** trên **Ví dụ 1**. Ghi lại lúc nào "quét đoạn mới" xảy ra.

```
      c0 c1 c2 c3
r0:   0  E  0  0
r1:   E  0  W  E
r2:   0  E  0  0
```

| ô `(i,j)` | giá trị | quét đoạn hàng? | `row_hits` | quét đoạn cột? | `col_hits[j]` | ô trống? ứng viên |
|---|---|---|---|---|---|---|
| `(0,0)` | `0` | có (`j = 0`): đoạn `0 E 0 0` | `1` | có (`i = 0`): đoạn `0 E 0` | `1` | `1 + 1 = 2` |
| `(0,1)` | `E` | không | `1` | có: cột 1 `E 0 E` | `2` | — |
| `(0,2)` | `0` | không | `1` | có: `0` rồi gặp `W` | `0` | `1 + 0 = 1` |
| `(0,3)` | `0` | không | `1` | có: `0 E 0` | `1` | `1 + 1 = 2` |
| `(1,0)` | `E` | có (`j = 0`): `E 0` rồi `W` | `1` | không (trên là `0`) | `1` | — |
| `(1,1)` | `0` | không | `1` | không | `2` | `1 + 2 = 3` ⭐ |
| `(1,2)` | `W` | không | `1` | không | `0` | — |
| `(1,3)` | `E` | có (trái là `W`): `E` | `1` | không | `1` | — |
| `(2,0)` | `0` | có (`j = 0`): `0 E 0 0` | `1` | không | `1` | `1 + 1 = 2` |
| `(2,1)` | `E` | không | `1` | không | `2` | — |
| `(2,2)` | `0` | không | `1` | có (trên là `W`): `0` | `0` | `1 + 0 = 1` |
| `(2,3)` | `0` | không | `1` | không | `1` | `1 + 1 = 2` |

`best = 3`. Hàm `trace_cached` dưới đây in đúng các bước này.

In [18]:
def trace_cached(grid):
    m, n = len(grid), len(grid[0])
    row_hits = 0
    col_hits = [0] * n
    best = 0
    for i in range(m):
        for j in range(n):
            notes = []
            if j == 0 or grid[i][j - 1] == 'W':
                row_hits = 0
                k = j
                while k < n and grid[i][k] != 'W':
                    row_hits += grid[i][k] == 'E'
                    k += 1
                notes.append(f"quét đoạn hàng [{j}..{k - 1}] -> row_hits = {row_hits}")
            if i == 0 or grid[i - 1][j] == 'W':
                col_hits[j] = 0
                k = i
                while k < m and grid[k][j] != 'W':
                    col_hits[j] += grid[k][j] == 'E'
                    k += 1
                notes.append(f"quét đoạn cột [{i}..{k - 1}] -> col_hits[{j}] = {col_hits[j]}")
            line = f"({i},{j}) '{grid[i][j]}' row_hits={row_hits} col_hits[{j}]={col_hits[j]}"
            if grid[i][j] == '0':
                cand = row_hits + col_hits[j]
                best = max(best, cand)
                line += f" -> ô trống: ứng viên {cand}, best = {best}"
            print(line + ("   | " + "; ".join(notes) if notes else ""))
    print("kết quả:", best)
    return best


assert trace_cached(EX1) == 3

(0,0) '0' row_hits=1 col_hits[0]=1 -> ô trống: ứng viên 2, best = 2   | quét đoạn hàng [0..3] -> row_hits = 1; quét đoạn cột [0..2] -> col_hits[0] = 1
(0,1) 'E' row_hits=1 col_hits[1]=2   | quét đoạn cột [0..2] -> col_hits[1] = 2
(0,2) '0' row_hits=1 col_hits[2]=0 -> ô trống: ứng viên 1, best = 2   | quét đoạn cột [0..0] -> col_hits[2] = 0
(0,3) '0' row_hits=1 col_hits[3]=1 -> ô trống: ứng viên 2, best = 2   | quét đoạn cột [0..2] -> col_hits[3] = 1
(1,0) 'E' row_hits=1 col_hits[0]=1   | quét đoạn hàng [0..1] -> row_hits = 1
(1,1) '0' row_hits=1 col_hits[1]=2 -> ô trống: ứng viên 3, best = 3
(1,2) 'W' row_hits=1 col_hits[2]=0
(1,3) 'E' row_hits=1 col_hits[3]=1   | quét đoạn hàng [3..3] -> row_hits = 1
(2,0) '0' row_hits=1 col_hits[0]=1 -> ô trống: ứng viên 2, best = 3   | quét đoạn hàng [0..3] -> row_hits = 1
(2,1) 'E' row_hits=1 col_hits[1]=2
(2,2) '0' row_hits=1 col_hits[2]=0 -> ô trống: ứng viên 1, best = 3   | quét đoạn cột [2..2] -> col_hits[2] = 0
(2,3) '0' row_hits=1 col_hits[3]

### Trace Cách 4 (`split`) cho một hàng và một cột của Ví dụ 1

Hàng `1`: `"E0WE"` → `split('W')` = `['E0', 'E']` → số `E` mỗi đoạn `[1, 1]` → trải về ô: `[1, 1, 0, 1]` (ô `W` nhận `0`). Cột `1`: `"E0E"` → không có `W` → một đoạn có `2` kẻ địch → `[2, 2, 2]`. Ô `(1, 1)`: `rows_kill[1][1] + cols_kill[1][1] = 1 + 2 = 3` ✔.

In [19]:
row1 = "".join(EX1[1])
col1 = "".join(list(zip(*EX1))[1])
print(repr(row1), "->", row1.split('W'), "->", seg_counts(row1))
print(repr(col1), "->", col1.split('W'), "->", seg_counts(col1))
print("ô (1,1):", seg_counts(row1)[1] + seg_counts(col1)[1])
assert seg_counts(row1) == [1, 1, 0, 1] and seg_counts(col1) == [2, 2, 2]

'E0WE' -> ['E0', 'E'] -> [1, 1, 0, 1]
'E0E' -> ['E0E'] -> [2, 2, 2]
ô (1,1): 3


## 9. Code nộp LeetCode

Mình nộp **Cách 3** (đếm tái sử dụng): `O(mn)` thời gian, `O(n)` bộ nhớ thêm, là cách chuẩn của bài.

In [20]:
from typing import List


class Solution:
    def maxKilledEnemies(self, grid: List[List[str]]) -> int:
        m, n = len(grid), len(grid[0])
        row_hits = 0                      # số E trong đoạn hàng hiện tại
        col_hits = [0] * n                # số E trong đoạn cột hiện tại của từng cột
        best = 0
        for i in range(m):
            for j in range(n):
                if j == 0 or grid[i][j - 1] == 'W':          # đầu đoạn hàng
                    row_hits = 0
                    k = j
                    while k < n and grid[i][k] != 'W':
                        row_hits += grid[i][k] == 'E'
                        k += 1
                if i == 0 or grid[i - 1][j] == 'W':          # đầu đoạn cột
                    col_hits[j] = 0
                    k = i
                    while k < m and grid[k][j] != 'W':
                        col_hits[j] += grid[k][j] == 'E'
                        k += 1
                if grid[i][j] == '0':                         # chỉ đặt bom ở ô trống
                    best = max(best, row_hits + col_hits[j])
        return best


sol = Solution()
for g, expected in examples:
    print(sol.maxKilledEnemies(g), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )


## 10. Vì sao thuật toán đúng

- **Tầm nổ là hợp của đoạn hàng và đoạn cột:** bom ở ô trống `(i, j)` diệt các `E` trên hàng `i` từ `(i, j)` đi trái/phải cho đến khi gặp tường — chính là các `E` của **đoạn hàng** chứa `(i, j)` (đoạn tối đại không chứa `W`). Tương tự cột. Hai đoạn chỉ giao nhau tại `(i, j)` (là ô trống, không phải `E`) nên **không đếm trùng**. Vậy số diệt được `= E(đoạn hàng) + E(đoạn cột)`.
- **`row_hits` đúng:** ta tính lại `row_hits` đúng khi bắt đầu một đoạn hàng mới: ở cột `0`, hoặc khi ô bên trái là `'W'`. Trong đoạn, mọi ô có cùng đoạn nên `row_hits` không đổi. Nếu ô `(i, j)` là `'W'` thì ô kế tiếp bắt đầu đoạn mới ⇒ được tính lại. (Ô ngay sau nhiều `W` liên tiếp cũng vậy: điều kiện chỉ nhìn ô bên trái.)
- **`col_hits[j]` đúng:** tương tự theo cột; vì ta duyệt theo hàng, `col_hits[j]` được **giữ** qua các hàng `i` trong cùng đoạn cột và chỉ **tính lại** khi `i == 0` hoặc ô trên là `'W'`.
- **Độ phức tạp:** mỗi ô thuộc đúng một đoạn hàng và một đoạn cột; ta quét mỗi đoạn một lần (ở ô đầu) nên tổng số bước quét `<= 2mn`.
- **Chỉ ô trống:** ta chỉ cập nhật `best` khi `grid[i][j] == '0'`. Nếu không có ô trống nào thì `best = 0` (khởi tạo `0`).
- **Các cách khác** tính cùng đại lượng `E(đoạn hàng) + E(đoạn cột)` bằng bảng bốn hướng, `split`, hay cộng điểm từ kẻ địch; mục 12 đối chiếu chúng với oracle.

## 11. Phân tích Time / Space Complexity

Gọi `m × n` là kích thước lưới, `E` là số kẻ địch.

| Cách | Time | Space |
|---|---|---|
| 1. Quét bốn hướng cho mỗi ô trống | `O(m·n·(m + n))` | `O(1)` |
| 2. Bốn bảng DP | `O(m·n)` | `O(m·n)` |
| 3. Đếm tái sử dụng `row_hits`, `col_hits` | **`O(m·n)`** | **`O(n)`** |
| 4. `split('W')` hàng/cột | `O(m·n)` | `O(m·n)` |
| 5. Từ kẻ địch cộng điểm | `O(E·(m + n))` | `O(m·n)` |

Với `m = n = 500` (`250 000` ô): Cách 2–4 chỉ khoảng `10^6` thao tác; Cách 1 khoảng `2.5 · 10^8` — không dùng được.

## 12. Bộ test

**Oracle độc lập** `oracle_kills(grid)`: với mỗi ô trống, dùng **cắt chuỗi** để đếm: phần bên trái của hàng (`row[:j]`), lấy đoạn sau tường cuối cùng (`rsplit('W', 1)[-1]`), phần bên phải (`row[j + 1:]`, lấy đoạn trước tường đầu tiên `split('W', 1)[0]`), và tương tự cho cột. Khác hẳn việc đi từng ô theo hướng của Cách 1 và DP.

In [21]:
def oracle_kills(grid):
    m, n = len(grid), len(grid[0])
    best = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] != '0':
                continue
            row = "".join(grid[i])
            col = "".join(grid[x][j] for x in range(m))
            total = (row[:j].rsplit('W', 1)[-1].count('E') + row[j + 1:].split('W', 1)[0].count('E')
                     + col[:i].rsplit('W', 1)[-1].count('E') + col[i + 1:].split('W', 1)[0].count('E'))
            best = max(best, total)
    return best


METHODS = [max_killed_bruteforce, max_killed_dp, max_killed_cached, max_killed_split, max_killed_from_enemies,
           lambda g: Solution().maxKilledEnemies(g),
           lambda g: SolutionDP().maxKilledEnemies(g),
           lambda g: SolutionSplit().maxKilledEnemies(g),
           lambda g: SolutionFromEnemies().maxKilledEnemies(g)]


def check(grid, expected=None):
    want = oracle_kills(grid) if expected is None else expected
    assert want == oracle_kills(grid), ("oracle", grid)
    for idx, f in enumerate(METHODS):
        got = f([row[:] for row in grid])
        assert got == want, (idx, grid, got, want)
    return want


def G(*rows):
    return [list(r) for r in rows]


# Ví dụ đề
assert check(EX1, 3) == 3
assert check(EX2, 1) == 1

# Biên: 1x1
assert check(G("0"), 0) == 0                 # ô trống nhưng không có địch
assert check(G("E"), 0) == 0                 # không có ô trống
assert check(G("W"), 0) == 0

# Không có ô trống nào / toàn tường / toàn địch
assert check(G("EEE", "EEE"), 0) == 0
assert check(G("WWW", "WWW"), 0) == 0
assert check(G("EWE", "WEW", "EWE"), 0) == 0

# Không có kẻ địch
assert check(G("000", "0W0", "000"), 0) == 0

# Một hàng / một cột
assert check(G("0E0E0"), 2) == 2             # bom ở ô trống bất kỳ: diệt cả 2 E (không tường)
assert check(G("0E0WE0"), 1) == 1            # tường chia hàng: đoạn trái 1 E, đoạn phải 1 E
assert check(G("E", "0", "E", "W", "E", "0"), 2) == 2
assert check(G("E", "W", "0"), 0) == 0

# Tường chặn: kẻ địch sau tường không bị diệt
assert check(G("0WE"), 0) == 0
assert check(G("0EWE"), 1) == 1

# Hàng + cột cùng góp: chữ thập
assert check(G("0E0", "E0E", "0E0"), 4) == 4    # ô giữa diệt 4 địch ở bốn phía
assert check(G("E0E", "000", "E0E"), 2) == 2    # bom ở (0,1): trái và phải đều là E; không có E trong cột
print("test cơ bản qua")

test cơ bản qua


### 12.2. Phản ví dụ cho các cách làm sai

Các hàm **sai** (mỗi hàm là một lỗi ở mục 13):

- `bad_ignore_wall`: quên tường chặn (đếm cả hàng/cột).
- `bad_bomb_anywhere`: cho phép đặt bom ở ô `'E'`.
- `bad_no_reset`: không reset `row_hits` ở tường (chỉ reset ở đầu hàng).
- `bad_int_zero`: so sánh ô với số `0` thay vì `'0'`.
- `bad_two_dirs`: chỉ tính phải và xuống.

In [22]:
def bad_ignore_wall(grid):
    m, n = len(grid), len(grid[0])
    best = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == '0':
                total = sum(c == 'E' for c in grid[i]) + sum(grid[x][j] == 'E' for x in range(m))
                best = max(best, total)
    return best


def bad_bomb_anywhere(grid):
    m, n = len(grid), len(grid[0])
    best = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] != 'W':                     # SAI: cho đặt bom cả ở ô E
                best = max(best, kills_at(grid, i, j))
    return best


def bad_no_reset(grid):
    m, n = len(grid), len(grid[0])
    row_hits, col_hits, best = 0, [0] * n, 0
    for i in range(m):
        for j in range(n):
            if j == 0:                                # SAI: chỉ tính lại ở đầu hàng, không xét tường
                row_hits = 0
                for k in range(n):
                    if grid[i][k] == 'E':
                        row_hits += 1
            if i == 0 or grid[i - 1][j] == 'W':
                col_hits[j] = 0
                k = i
                while k < m and grid[k][j] != 'W':
                    col_hits[j] += grid[k][j] == 'E'
                    k += 1
            if grid[i][j] == '0':
                best = max(best, row_hits + col_hits[j])
    return best


def bad_int_zero(grid):
    m, n = len(grid), len(grid[0])
    best = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 0:                       # SAI: so sánh với số 0, không bao giờ đúng
                best = max(best, kills_at(grid, i, j))
    return best


def bad_two_dirs(grid):
    m, n = len(grid), len(grid[0])
    best = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == '0':
                total = sum(1 for d in [(0, 1), (1, 0)] for (x, y) in ray(grid, i, j, *d) if grid[x][y] == 'E')
                best = max(best, total)
    return best


for bad in (bad_ignore_wall, bad_bomb_anywhere, bad_no_reset, bad_int_zero, bad_two_dirs):
    print(f"{bad.__name__:18s} trên Ví dụ 1 ->", bad(EX1), " Ví dụ 2 ->", bad(EX2), "(đúng 3 và 1)")
assert bad_ignore_wall(EX1) != 3 or bad_ignore_wall(EX2) != 1
assert bad_int_zero(EX1) == 0
assert bad_two_dirs(EX1) != 3
g_e = G("EEE")                                  # không có ô trống: đúng là 0
print("bom ở ô E:", bad_bomb_anywhere(g_e), "| đúng:", Solution().maxKilledEnemies(g_e))
g_nr = G("EW0")                                 # ô trống (0,2) nằm sau tường: đoạn của nó không có E
print("không reset ở tường:", bad_no_reset(g_nr), "| đúng:", Solution().maxKilledEnemies(g_nr))
assert bad_no_reset(g_nr) != Solution().maxKilledEnemies(g_nr)

bad_ignore_wall    trên Ví dụ 1 -> 4  Ví dụ 2 -> 1 (đúng 3 và 1)
bad_bomb_anywhere  trên Ví dụ 1 -> 3  Ví dụ 2 -> 2 (đúng 3 và 1)
bad_no_reset       trên Ví dụ 1 -> 4  Ví dụ 2 -> 1 (đúng 3 và 1)
bad_int_zero       trên Ví dụ 1 -> 0  Ví dụ 2 -> 0 (đúng 3 và 1)
bad_two_dirs       trên Ví dụ 1 -> 2  Ví dụ 2 -> 1 (đúng 3 và 1)
bom ở ô E: 2 | đúng: 0
không reset ở tường: 1 | đúng: 0


### 12.3. Đối chiếu ngẫu nhiên (có seed) và ca lớn

Sinh lưới `m × n` ngẫu nhiên với `m, n` từ `1` đến `9`, mỗi ô là `'W'`, `'E'` hoặc `'0'` với **mật độ khác nhau** (nhiều tường, ít tường, nhiều địch...). So **mọi cách** với `oracle_kills`.

In [23]:
rng = random.Random(361)
stat = Counter()
for case in range(4000):
    m, n = rng.randint(1, 9), rng.randint(1, 9)
    weights = rng.choice([(1, 1, 1), (3, 1, 1), (1, 3, 1), (1, 1, 3), (1, 4, 4), (0, 1, 1)])   # (W, E, 0)
    grid = [[rng.choices("WE0", weights)[0] for _ in range(n)] for _ in range(m)]
    want = check(grid)
    stat[min(want, 6)] += 1
print("4000 lưới ngẫu nhiên: mọi cách khớp oracle; phân bố đáp án (6 = từ 6 trở lên):", dict(sorted(stat.items())))
assert len(stat) >= 5

4000 lưới ngẫu nhiên: mọi cách khớp oracle; phân bố đáp án (6 = từ 6 trở lên): {0: 494, 1: 450, 2: 499, 3: 549, 4: 489, 5: 406, 6: 1113}


In [24]:
# Ca lớn 500 x 500 (đúng giới hạn): so các cách O(mn) với nhau (Cách 1 quá chậm nên không chạy)
rng = random.Random(3610)
for name, weights in [("nhiều trống", (1, 2, 20)), ("nhiều địch", (2, 20, 2)), ("không tường", (0, 1, 1))]:
    big = [[rng.choices("WE0", weights)[0] for _ in range(500)] for _ in range(500)]
    t0 = time.perf_counter()
    r3 = Solution().maxKilledEnemies(big)
    t3 = time.perf_counter() - t0
    t0 = time.perf_counter()
    r2 = max_killed_dp(big)
    t2 = time.perf_counter() - t0
    t0 = time.perf_counter()
    r4 = max_killed_split(big)
    t4 = time.perf_counter() - t0
    assert r2 == r3 == r4, (r2, r3, r4)
    print(f"{name:12s}: đáp án {r3}; Cách 3 {t3:.2f}s, Cách 2 {t2:.2f}s, Cách 4 {t4:.2f}s")

nhiều trống : đáp án 47; Cách 3 0.08s, Cách 2 0.16s, Cách 4 0.03s


nhiều địch  : đáp án 173; Cách 3 0.08s, Cách 2 0.11s, Cách 4 0.03s


không tường : đáp án 566; Cách 3 0.09s, Cách 2 0.12s, Cách 4 0.03s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên tường chặn

**Code sai.** Đếm `E` của **cả hàng** và **cả cột**.
```python
total = sum(c == 'E' for c in grid[i]) + sum(grid[x][j] == 'E' for x in range(m))
```
**Vì sao sai.** Tường `'W'` chặn bom. Ví dụ 1: ở ô `(1,1)` bên phải là `W` — nhưng `E` ở `(1,3)` nằm sau tường, không bị diệt.

**Code đúng.** Chỉ đếm trong **đoạn** giữa hai tường (quét tới tường rồi dừng).

In [25]:
print("sai :", bad_ignore_wall(EX1), "(đúng 3)")
print("đúng:", Solution().maxKilledEnemies(EX1))

sai : 4 (đúng 3)
đúng: 3


### Lỗi 2: Đặt bom ở ô `'E'` (hoặc `'W'`)

**Code sai.** Lấy tối đa trên **mọi** ô không phải tường.

**Vì sao sai.** Đề: bom **chỉ đặt vào ô trống**. Ở lưới một hàng `["EEE"]` không có ô trống nào ⇒ đáp án `0`; cách sai đặt bom ở `E` giữa và diệt `2`.

**Code đúng.** `if grid[i][j] == '0':` trước khi cập nhật `best`.

In [26]:
print("sai :", bad_bomb_anywhere(g_e), "| đúng:", Solution().maxKilledEnemies(g_e))

sai : 2 | đúng: 0


### Lỗi 3: Không reset bộ đếm tại tường

**Code sai.** Chỉ tính `row_hits` ở đầu hàng (`j == 0`), không tính lại sau mỗi `'W'`.

**Vì sao sai.** Sau một bức tường, các ô bên phải thuộc **đoạn mới**; đoạn mới có số `E` khác. Dùng lại `row_hits` của đoạn trước cho kết quả sai (cả nếu đoạn mới ít hơn lẫn nhiều hơn).

**Code đúng.** Điều kiện tính lại: `j == 0 or grid[i][j - 1] == 'W'`.

In [27]:
print("sai :", bad_no_reset(g_nr), "| đúng:", Solution().maxKilledEnemies(g_nr))

sai : 1 | đúng: 0


### Lỗi 4: So sánh ô với số `0` thay vì ký tự `'0'`

**Code sai.**
```python
if grid[i][j] == 0:
```
**Vì sao sai.** Ô là **chuỗi** `'0'`; `'0' == 0` là `False` ⇒ không ô nào được coi là ô trống ⇒ luôn trả `0`.

**Code đúng.** `grid[i][j] == '0'`.

In [28]:
print("sai :", bad_int_zero(EX1), "| đúng:", Solution().maxKilledEnemies(EX1))

sai : 0 | đúng: 3


### Lỗi 5: Chỉ đếm hai trong bốn hướng

**Code sai.** Chỉ đếm bên phải và bên dưới (nghĩ "bom nổ về phía trước").

**Vì sao sai.** Đề: kẻ địch **cùng hàng và cùng cột**, cả hai phía. Ví dụ 1 ở `(1,1)` cần `E` bên trái `(1,0)` và `E` ở phía trên `(0,1)`.

**Code đúng.** Bốn hướng (hoặc đoạn hàng + đoạn cột trọn vẹn).

In [29]:
print("sai :", bad_two_dirs(EX1), "| đúng:", Solution().maxKilledEnemies(EX1))

sai : 2 | đúng: 3


### Lỗi 6 (bonus): khởi tạo `best` sai khi không có ô trống

**Code sai.** `best = float('-inf')` hoặc `max([...])` trên danh sách rỗng.

**Vì sao sai.** Lưới có thể không có ô trống (ví dụ `[["E"]]`); khi đó `best` giữ `-inf` hoặc `max([])` báo `ValueError`. Đề kỳ vọng `0` (không đặt được bom nên diệt `0` — đối chiếu lại đề gốc nếu có Premium).

**Code đúng.** `best = 0` hoặc `max(..., default=0)`.

In [30]:
def bad_neg_inf(grid):
    m, n = len(grid), len(grid[0])
    return max(kills_at(grid, i, j) for i in range(m) for j in range(n) if grid[i][j] == '0')

try:
    bad_neg_inf([["E"]])
except ValueError as e:
    print("sai: ValueError:", e)
print("đúng:", Solution().maxKilledEnemies([["E"]]))

sai: ValueError: max() iterable argument is empty
đúng: 0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "mỗi ô chịu ảnh hưởng của cả hàng và cột, bị chặn bởi chướng ngại" ⇒ nghĩ ngay tới **đoạn** (segment) giữa hai chướng ngại; kết quả = tổng hai đoạn (hàng + cột).
- Thay vì quét lại từng ô: **đếm một lần ở đầu đoạn**, dùng lại cho cả đoạn — chuyển `O(mn(m + n))` thành `O(mn)`.
- Có hai khuôn: **bốn bảng DP** (dễ hiểu, `O(mn)` bộ nhớ) và **đếm tái sử dụng** (gọn, `O(n)` bộ nhớ). Học cả hai.
- Với Python, thao tác chuỗi `split`/`count` viết bằng C nên rất nhanh; `zip(*grid)` giúp xử lý cột như hàng.
- Nhớ **ký tự `'0'`**, không phải số `0`; và chỉ đặt bom ở ô trống.
- Đảo vai (từ kẻ địch cộng điểm cho ô trống) là mẹo chung: "tính hộ cho nhiều ô cùng lúc từ phía nguồn".
- Khởi tạo kết quả `0` để xử lý trường hợp không có ô trống.
- Tự kiểm tra bằng ví dụ nhỏ có tường ở giữa hàng, tường ở biên, và hai tường liên tiếp.

## 15. Biến thể của bài

1. **Tầm nổ giới hạn `k`:** bom chỉ diệt trong `k` ô mỗi hướng (vẫn bị tường chặn). Cần **cửa sổ** thay vì đoạn: có thể dùng tổng tiền tố theo hàng/cột (hàm `max_killed_range` bên dưới).
2. **Kẻ địch chặn bom:** bom chỉ diệt **kẻ địch đầu tiên** mỗi hướng (như "tầm nhìn"). Mỗi hướng đóng góp tối đa `1`.
3. **Trả về vị trí** tốt nhất (hàng, cột) và số lượng.
4. **Nhiều quả bom:** chọn `2` ô để tối đa hóa tổng, **không đếm trùng** — khó hơn nhiều.
5. **Đếm kẻ địch bị nhiều bom bắn tới / cell có thể bị nhìn thấy:** bài liên quan tới "tầm nhìn".
6. **Bài cùng họ (đoạn/hàng/cột, tiền tố 2D):** LeetCode 304 (Range Sum Query 2D), 221 (Maximal Square), 51 (N-Queens: tấn công theo hàng/cột/chéo), 200 (Number of Islands).

In [31]:
def max_killed_range(grid, k):
    """Biến thể: bom chỉ nổ tối đa k ô mỗi hướng (vẫn bị tường chặn). Cài bằng quét trực tiếp (oracle cho k nhỏ)."""
    m, n = len(grid), len(grid[0])
    best = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] != '0':
                continue
            total = 0
            for di, dj in DIRS:
                x, y = i + di, j + dj
                steps = 0
                while steps < k and 0 <= x < m and 0 <= y < n and grid[x][y] != 'W':
                    total += grid[x][y] == 'E'
                    x += di
                    y += dj
                    steps += 1
            best = max(best, total)
    return best


def best_position(grid):
    """Trả (số diệt, hàng, cột) của ô trống tốt nhất (ô đầu tiên theo thứ tự hàng rồi cột); (0, -1, -1) nếu không có ô trống."""
    best = (0, -1, -1)
    for i in range(len(grid)):
        for j in range(len(grid[0])):
            if grid[i][j] == '0':
                v = kills_at(grid, i, j)
                if best[1] == -1 or v > best[0]:
                    best = (v, i, j)
    return best


print(max_killed_range(EX1, 1), max_killed_range(EX1, 10), best_position(EX1), best_position([["E"]]))
assert max_killed_range(EX1, 10) == Solution().maxKilledEnemies(EX1) == 3
assert max_killed_range(EX1, 1) == 3                      # (1,1): 3 hàng xóm trực tiếp đều là E
assert best_position(EX1) == (3, 1, 1)
assert best_position([["E"]]) == (0, -1, -1)
rng = random.Random(15)
for _ in range(300):
    g = [[rng.choice("WE0") for _ in range(6)] for _ in range(rng.randint(1, 6))]
    assert max_killed_range(g, 100) == oracle_kills(g)

3 3 (3, 1, 1) (0, -1, -1)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét bốn hướng mỗi ô trống | trực tiếp theo đề | `O(m·n·(m + n))` | `O(1)` | hiểu đề, oracle cho lưới nhỏ |
| 2. Bốn bảng DP | `left/right/up/down` bằng truy hồi | `O(m·n)` | `O(m·n)` | dễ hiểu, dễ chứng minh |
| 3. Đếm tái sử dụng | `row_hits`, `col_hits[j]` tính lại ở đầu đoạn | `O(m·n)` | `O(n)` | **nên nộp**; tiết kiệm bộ nhớ |
| 4. `split('W')` | cắt hàng/cột thành đoạn, đếm `E` mỗi đoạn | `O(m·n)` | `O(m·n)` | gọn, nhanh nhờ hàm chuỗi |
| 5. Từ kẻ địch cộng điểm | đảo vai: nguồn → đích | `O(E·(m + n))` | `O(m·n)` | khi kẻ địch thưa |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"0EWE0".split('W')`, `'0' == 0`, số kẻ địch diệt được nếu đặt bom ở `(0, 0)` của lưới `[["0","E"],["E","W"]]` (không dùng thuật toán; chỉ nhìn hàng và cột).

**Bài 2 (tính tay).** Với lưới `[["0","E","0"],["E","W","E"],["0","E","0"]]`: lập bảng cho mỗi ô trống (hàng + cột) rồi cho biết đáp án.

**Bài 3 (điền TODO).** Hoàn thành `row_segment_counts(row)` (trả list số `E` của đoạn chứa mỗi ô, ô `'W'` nhận `0`) bằng cách đếm cộng dồn hai lượt: lượt xuôi tính `E` bên trái (reset ở tường), lượt ngược tính `E` bên phải; kết quả mỗi ô = trái + phải (**không** tính chính ô đó — ô `'E'` thì cộng thêm `1`).

In [32]:
def row_segment_counts(row):
    # TODO 1: lượt xuôi: cnt = 0; với mỗi j: nếu row[j] == 'W' thì cnt = 0; left[j] = cnt; sau đó nếu row[j] == 'E' thì cnt += 1
    # TODO 2: lượt ngược (j từ n-1 về 0): làm tương tự để có right[j]
    # TODO 3: kết quả ô j = 0 nếu row[j] == 'W'; ngược lại left[j] + right[j] + (1 nếu row[j] == 'E')
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết lại **từ đầu** `max_killed_mine(grid)` theo Cách 3 (đếm tái sử dụng) mà không nhìn code mẫu; kiểm tra với hai ví dụ đề và với `oracle_kills` trên 200 lưới ngẫu nhiên.

**Bài 5 (biến thể / mini challenge).** (a) Viết `kill_matrix(grid)` trả ma trận cùng kích thước: ô trống chứa số diệt được nếu đặt bom ở đó, ô khác chứa `-1`. (b) Viết `max_killed_first_only(grid)` cho biến thể "bom chỉ diệt **kẻ địch đầu tiên** gặp ở mỗi hướng (trước khi gặp tường)" — mỗi hướng đóng góp `0` hoặc `1`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `['0E', 'E0']`; `False`; ô `(0,0)`: hàng `0`: bên phải `E` ⇒ `1`; cột `0`: bên dưới `E` ⇒ `1` ⇒ tổng `2`. Bài 3: hai lượt, mỗi lượt reset ở `'W'`; cộng thêm `1` nếu chính ô đó là `E`.
- **Mức 2 (vừa):** Bài 2: các ô trống là bốn góc `(0,0)`, `(0,2)`, `(2,0)`, `(2,2)`. Ô `(0,0)`: hàng `0` (`0 E 0`) có `1` địch; cột `0` (`0, E, 0`) có `1` địch ⇒ `2`. Bốn góc đối xứng nhau nên đều bằng `2`. Đáp án `2`.
- **Mức 3 (sâu):** Bài 4: `row_hits` tính lại khi `j == 0 or grid[i][j-1] == 'W'`; `col_hits[j]` khi `i == 0 or grid[i-1][j] == 'W'`; chỉ cập nhật `best` ở ô `'0'`. Bài 5(b): với mỗi hướng, đi từ ô kế tiếp: gặp `'W'` dừng (`+0`); gặp `'E'` thì `+1` rồi dừng; gặp `'0'` đi tiếp.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert "0EWE0".split('W') == ['0E', 'E0']
assert ('0' == 0) is False
assert kills_at([["0", "E"], ["E", "W"]], 0, 0) == 2

# Bài 2
g2 = [["0", "E", "0"], ["E", "W", "E"], ["0", "E", "0"]]
assert [kills_at(g2, i, j) for i, j in [(0, 0), (0, 2), (2, 0), (2, 2)]] == [2, 2, 2, 2]
assert Solution().maxKilledEnemies(g2) == 2


# Bài 3
def row_segment_counts_ans(row):
    n = len(row)
    left = [0] * n
    right = [0] * n
    cnt = 0
    for j in range(n):                      # lượt xuôi
        if row[j] == 'W':
            cnt = 0
        left[j] = cnt
        if row[j] == 'E':
            cnt += 1
    cnt = 0
    for j in range(n - 1, -1, -1):          # lượt ngược
        if row[j] == 'W':
            cnt = 0
        right[j] = cnt
        if row[j] == 'E':
            cnt += 1
    return [0 if row[j] == 'W' else left[j] + right[j] + (row[j] == 'E') for j in range(n)]


for s in ["0E0W0EE0", "W0W", "EE", "0", "E", "WW", "0E0E0"]:
    assert row_segment_counts_ans(list(s)) == seg_counts(s), s


# Bài 4
def max_killed_mine(grid):
    m, n = len(grid), len(grid[0])
    row_hits, col_hits, best = 0, [0] * n, 0
    for i in range(m):
        for j in range(n):
            if j == 0 or grid[i][j - 1] == 'W':
                row_hits = 0
                k = j
                while k < n and grid[i][k] != 'W':
                    row_hits += grid[i][k] == 'E'
                    k += 1
            if i == 0 or grid[i - 1][j] == 'W':
                col_hits[j] = 0
                k = i
                while k < m and grid[k][j] != 'W':
                    col_hits[j] += grid[k][j] == 'E'
                    k += 1
            if grid[i][j] == '0':
                best = max(best, row_hits + col_hits[j])
    return best


for g, expected in examples:
    assert max_killed_mine(g) == expected
rng = random.Random(19)
for _ in range(200):
    n_cols = rng.randint(1, 8)
    g = [[rng.choice("WE0") for _ in range(n_cols)] for _ in range(rng.randint(1, 8))]
    assert max_killed_mine(g) == oracle_kills(g)


# Bài 5
def kill_matrix(grid):
    return [[kills_at(grid, i, j) if grid[i][j] == '0' else -1 for j in range(len(grid[0]))] for i in range(len(grid))]


def max_killed_first_only(grid):
    m, n = len(grid), len(grid[0])
    best = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] != '0':
                continue
            total = 0
            for di, dj in DIRS:
                x, y = i + di, j + dj
                while 0 <= x < m and 0 <= y < n and grid[x][y] == '0':
                    x += di
                    y += dj
                if 0 <= x < m and 0 <= y < n and grid[x][y] == 'E':
                    total += 1                      # kẻ địch đầu tiên trước khi gặp tường
            best = max(best, total)
    return best


km = kill_matrix(EX1)
assert km == [[2, -1, 1, 2], [-1, 3, -1, -1], [2, -1, 1, 2]]
assert max(max(r) for r in km) == 3
assert max_killed_first_only(EX1) == 3             # (1,1): trái E, trên E, dưới E (phải là W)
assert max_killed_first_only(G("0E0E0")) == 1 + 1  # ô giữa: trái E, phải E
assert max_killed_first_only(G("EE0")) == 1        # chỉ E đầu tiên gần nhất
print("Tất cả đáp án bài tập đều đúng")

Tất cả đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm tái sử dụng: `row_hits`, `col_hits[j]`)

```python
class Solution:
    def maxKilledEnemies(self, grid: List[List[str]]) -> int:
        m, n = len(grid), len(grid[0])
        row_hits, col_hits, best = 0, [0] * n, 0
        for i in range(m):
            for j in range(n):
                if j == 0 or grid[i][j - 1] == 'W':          # đầu đoạn hàng
                    row_hits = 0
                    k = j
                    while k < n and grid[i][k] != 'W':
                        row_hits += grid[i][k] == 'E'
                        k += 1
                if i == 0 or grid[i - 1][j] == 'W':          # đầu đoạn cột
                    col_hits[j] = 0
                    k = i
                    while k < m and grid[k][j] != 'W':
                        col_hits[j] += grid[k][j] == 'E'
                        k += 1
                if grid[i][j] == '0':
                    best = max(best, row_hits + col_hits[j])
        return best
```

**Checklist**

- [ ] Tường `'W'` **chặn** bom (chỉ đếm `E` trong đoạn giữa hai tường)?
- [ ] Bom đặt **chỉ ở ô `'0'`** (ký tự, không phải số `0`)?
- [ ] `row_hits` tính lại khi `j == 0` hoặc ô bên **trái** là `'W'`; `col_hits[j]` khi `i == 0` hoặc ô **trên** là `'W'`?
- [ ] Đã tính **cả bốn hướng** (đoạn hàng + đoạn cột trọn vẹn)?
- [ ] `best` khởi tạo `0` (không có ô trống ⇒ `0`)?
- [ ] Đã thử: Ví dụ 1 (`3`), Ví dụ 2 (`1`), lưới `1×1`, toàn `E`, toàn `W`, hai tường liên tiếp?
- [ ] Skeleton đúng `class Solution` / `def maxKilledEnemies(self, grid: List[List[str]]) -> int`?
- [ ] (Premium) đã đối chiếu lại đề gốc nếu có tài khoản?